In [21]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("spark_homework_week3_notebook") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "../../warehouse") \
    .getOrCreate()

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print(f"Automatic broadcast join disabled: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")

data_path = "../../data"

match_details = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv(f"{data_path}/match_details.csv")
# match_details.show(3)

# matches = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv(f"{data_path}/matches.csv")
# # matches.show(3)

# medals_matches_players = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv(f"{data_path}/medals_matches_players.csv")
# # medals_matches_players.show(3, truncate=False)

# medals = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv(f"{data_path}/medals.csv")
# # medals.show(3)

# maps = spark.read.option("header", "true").option("inferSchema", "true") \
#     .csv(f"{data_path}/maps.csv")
# # maps.show(3)
# print('Data loaded')

# Create Iceberg table with bucketing using DISTRIBUTE BY for hash distribution
match_details.write.mode("overwrite") \
    .option("write.distribution-mode", "hash") \
    .option("write.hash.buckets", "16") \
    .option("write.hash.columns", "match_id") \
    .saveAsTable("local.default.match_details_bucketed", format="iceberg")

25/08/06 23:18:20 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Automatic broadcast join disabled: -1


Py4JJavaError: An error occurred while calling o231.saveAsTable.
: org.apache.iceberg.exceptions.NoSuchTableException: Invalid table identifier: match_details_bucketed
	at org.apache.iceberg.rest.RESTSessionCatalog.checkIdentifierIsValid(RESTSessionCatalog.java:1161)
	at org.apache.iceberg.rest.RESTSessionCatalog$Builder.<init>(RESTSessionCatalog.java:795)
	at org.apache.iceberg.rest.RESTSessionCatalog.buildTable(RESTSessionCatalog.java:574)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.buildTable(BaseSessionCatalog.java:84)
	at org.apache.iceberg.rest.RESTCatalog.buildTable(RESTCatalog.java:112)
	at org.apache.iceberg.CachingCatalog$CachingTableBuilder.<init>(CachingCatalog.java:222)
	at org.apache.iceberg.CachingCatalog.buildTable(CachingCatalog.java:214)
	at org.apache.iceberg.spark.SparkCatalog.newBuilder(SparkCatalog.java:1007)
	at org.apache.iceberg.spark.SparkCatalog.stageCreateOrReplace(SparkCatalog.java:295)
	at org.apache.spark.sql.connector.catalog.StagingTable

In [18]:
# match_details.write.mode("overwrite").save(f"{data_path}/output/match_details_bucketed")

match_details.write.mode("overwrite").option("path", f"{data_path}/output") \
.bucketBy(16, "match_id").saveAsTable("match_details_bucketed")

Py4JJavaError: An error occurred while calling o186.saveAsTable.
: org.apache.iceberg.exceptions.NoSuchTableException: Invalid table identifier: match_details_bucketed
	at org.apache.iceberg.rest.RESTSessionCatalog.checkIdentifierIsValid(RESTSessionCatalog.java:1161)
	at org.apache.iceberg.rest.RESTSessionCatalog$Builder.<init>(RESTSessionCatalog.java:795)
	at org.apache.iceberg.rest.RESTSessionCatalog.buildTable(RESTSessionCatalog.java:574)
	at org.apache.iceberg.catalog.BaseSessionCatalog$AsCatalog.buildTable(BaseSessionCatalog.java:84)
	at org.apache.iceberg.rest.RESTCatalog.buildTable(RESTCatalog.java:112)
	at org.apache.iceberg.CachingCatalog$CachingTableBuilder.<init>(CachingCatalog.java:222)
	at org.apache.iceberg.CachingCatalog.buildTable(CachingCatalog.java:214)
	at org.apache.iceberg.spark.SparkCatalog.newBuilder(SparkCatalog.java:1007)
	at org.apache.iceberg.spark.SparkCatalog.stageCreateOrReplace(SparkCatalog.java:295)
	at org.apache.spark.sql.connector.catalog.StagingTable